In [5]:
import numpy as np 
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.integrate import simps
from scipy.special import ellipk, ellipe, elliprf, elliprj

In [3]:
def polar_roots(E, Lz, Q, k):
    """Function to calculate the roots of the polar equation: z_+ and z_-. G=c=1. From the paper: Hughes 2000 - Evolution of circular, equatorial orbits of Kerr black holes due to gravitational wave emission.
    - https://ui.adsabs.harvard.edu/abs/2000PhRvD..61h4004H/abstract - PhysRevD.61.084004 -- EQ 2.2
    ______________________________________________________________________________________
    Parameters:
    E: Energy of the orbit
    Lz: Angular momentum in the z direction of the orbit
    Q: Carter constant of the orbit
    k: spin parameter of the black hole (0<=k<=1)
    ______________________________________________________________________________________
    Returns:
    z_+, z_-: Roots of the polar equation
    ______________________________________________________________________________________
    """
    L_sqr = Q + Lz**2 # Total angular momentum squared
    
    beta = (k**2)*(1 - E**2) # as defined in the paper under EQ 2.2
    
    # Just using quadratic formula to find the roots
    
    z_plus = ((L_sqr + beta) + np.sqrt((L_sqr + beta)**2 - 4*Q*beta))/(2*beta)
    z_minus = ((L_sqr + beta) - np.sqrt((L_sqr + beta)**2 - 4*Q*beta))/(2*beta)
    
    return z_plus, z_minus
    

In [6]:
def polar_period(r, E, Lz, Q, k):
    """Function to calculate the period of the polar orbit. G=c=1. From the paper: Hughes 2000 - Evolution of circular, equatorial orbits of Kerr black holes due to gravitational wave emission.
    - https://ui.adsabs.harvard.edu/abs/2000PhRvD..61h4004H/abstract - PhysRevD.61.084004 -- EQ 2.14
    ______________________________________________________________________________________
    Parameters:
    r: Radial coordinate of the orbit
    E: Energy of the orbit
    Lz: Angular momentum in the z direction of the orbit
    Q: Carter constant of the orbit
    k: spin parameter of the black hole (0<=k<=1)
    ______________________________________________________________________________________
    Returns:
    T: Period of the polar orbit
    ______________________________________________________________________________________
    """
    z_plus, z_minus = polar_roots(E, Lz, Q, k) # roots of the polar equation from equation 2.2 in Hughes 2000
    
    sqrt_z_ratio = np.sqrt(z_minus/z_plus)
    
    Delta = r**2 - 2*r + k**2
    gamma = E*((((r**2 + k**2)**2)/Delta) - k**2) + k*Lz*(1 - (r**2 + k**2)/Delta) # Hughes 2000 - EQ 2.11
    
    beta = (k**2)*(1 - E**2) # as defined in the paper under EQ 2.2
    
    T_theta = (4*gamma/np.sqrt(beta*z_plus))*ellipk(sqrt_z_ratio) + 4*(k**2)*E/np.sqrt(z_plus/(beta))*(ellipk(sqrt_z_ratio) - ellipe(sqrt_z_ratio)) # Hughes 2000 - EQ 2.14
    return T_theta

def elliptic_integral_third_kind_inc(phi, n, k):
    """Function to calculate the incomplete elliptic integral of the third kind. Used in the next function couldn't find a built-in function for this.
    From Numerical Recipes in C: The Art of Scientific Computing: Press, Teukolsky, Vetterling, Flannery: 2nd Edition -- Section 6.11 -- Equation 6.11.21 -- Page 268
    ______________________________________________________________________________________
    Parameters:
    phi: Upper limit of the integral
    n: characteristic of the elliptic integral
    k: parameter of the integral
    ______________________________________________________________________________________
    Returns:
    EI_3rdk_inc: Elliptic integral of the third kind
    ______________________________________________________________________________________
    """
    EI_rf = elliprf((np.cos(phi))**2, 1 - ((k**2)*(np.sin(phi))**2), 1)
    EI_rj = elliprj((np.cos(phi))**2, 1 - ((k**2)*(np.sin(phi))**2), 1, 1 + n*(np.sin(phi))**2)
    
    EI_3rdk_inc = (np.sin(phi))*EI_rf - n*((np.sin(phi))**3)*EI_rj/3
    return EI_3rdk_inc

def azimuthal_period(r, E, Lz, Q, k):
    """Function to calculate the period of the azimuthal orbit. G=c=1. From the paper: Hughes 2000 - Evolution of circular, equatorial orbits of Kerr black holes due to gravitational wave emission.
    - https://ui.adsabs.harvard.edu/abs/2000PhRvD..61h4004H/abstract - PhysRevD.61.084004 -- EQ 2.22
    ______________________________________________________________________________________
    Parameters:
    r: Radial coordinate of the orbit
    E: Energy of the orbit
    Lz: Angular momentum in the z direction of the orbit
    Q: Carter constant of the orbit
    k: spin parameter of the black hole (0<=k<=1)
    ______________________________________________________________________________________
    Returns:
    T: Period of the azimuthal orbit
    ______________________________________________________________________________________
    """
    z_plus, z_minus = polar_roots(E, Lz, Q, k) # roots of the polar equation from equation 2.2 in Hughes 2000
    sqrt_z_ratio = np.sqrt(z_minus/z_plus)
    
    polar_period_value = polar_period(r, E, Lz, Q, k) # from equation 2.14 in Hughes 2000
    
    # now we calculate the azimuthal angle moved during the polar period
    
    Delta = r**2 - 2*r + k**2
    delta = k*E*((r**2 + k**2)/Delta - 1) - k**2*Lz/Delta # Hughes 2000 - EQ 2.16
    
    phi_polar_period = (4/np.sqrt(beta*z_plus))*(Lz*elliptic_integral_third_kind_inc(np.pi/2, -z_minus, sqrt_z_ratio) - delta*ellipk(sqrt_z_ratio)) # Hughes 2000 - EQ 2.19
    
    # now we calculate the azimuthal period
    
    T_phi = (2*np.pi/polar_period_value)*phi_polar_period # Hughes 2000 - EQ 2.22
    return T_phi